**Data Cleanup**

In [53]:
"""
Data we need
- set of exams to schedule
- list of timeslots
- list of rooms available, by timeslot
- set of unique student schedules
- number of students with each unique schedule
- where classes during the semester
- set of classes that need to be split across rooms

Hard Constraints:
- room capacities
- all exams scheduled


Factors to consider:
1. student overlaps
2. 3 exams in 48 hours
3. back-to-back exams
4. in-person exams finish as early as possible
5. accomodate room requests
6. correct room type

If we have time:
- minimize exams on sunday
- fairness across majors

"""

'\nData we need\n- set of exams to schedule\n- list of timeslots\n- list of rooms available, by timeslot\n- set of unique student schedules\n- number of students with each unique schedule\n- where classes during the semester\n- set of classes that need to be split across rooms\n\nHard Constraints:\n- room capacities\n- all exams scheduled\n\n\nFactors to consider:\n1. student overlaps\n2. 3 exams in 48 hours\n3. back-to-back exams\n4. in-person exams finish as early as possible\n5. accomodate room requests\n6. correct room type\n\nIf we have time:\n- minimize exams on sunday\n- fairness across majors\n\n'

In [54]:
import pandas as pd
from datetime import datetime

# Load the CSV files
class_info = pd.read_csv('/Users/benfreidinger/Desktop/decisionlab26/past_group_work/2023-2024/Class_Info2023.csv')
final_exams = pd.read_csv('/Users/benfreidinger/Desktop/decisionlab26/past_group_work/2023-2024/Final_Exams2023.csv')
schedule = pd.read_csv('/Users/benfreidinger/Desktop/decisionlab26/past_group_work/2023-2024/Schedule2023.csv')
student_reg = pd.read_csv('/Users/benfreidinger/Desktop/decisionlab26/past_group_work/2023-2024/StudentRegistration2023.csv')

# Process exams (E): list of CRNs from final_exams
E = final_exams['CRN'].tolist()
E = list(set(E))

# Class sizes: number of students per exam (CRN)
class_sizes = student_reg.groupby('CRN').size().to_dict()

# Timeslots (T): unique (EXAM_DATE, EXAM_TIME) from final_exams, sorted chronologically
timeslots = final_exams[['EXAM_DATE', 'EXAM_TIME']].drop_duplicates()
T = [(row['EXAM_DATE'], row['EXAM_TIME']) for _, row in timeslots.iterrows()]
T = sorted(T, key=lambda t: (datetime.strptime(t[0], '%m/%d/%Y'), t[1]))

# Rooms (R): unique rooms from class_info where AVAILABLE_TO_SCHEDULE == 'Y'
available_rooms = class_info[class_info['AVAILABLE_TO_SCHEDULE'] == 'Y']
rooms = available_rooms[['BUILDING_CODE', 'ROOM_NUMBER']].drop_duplicates()
R = [(row['BUILDING_CODE'], row['ROOM_NUMBER']) for _, row in rooms.iterrows()]

# Include dummy room pairs for oversized exams
R = R + [('dummy', '1'), ('dummy', '2'), ('dummy', '3')]

# Capacities: dict room -> ROOM_CAPACITY
capacities = {(row['BUILDING_CODE'], row['ROOM_NUMBER']): row['ROOM_CAPACITY'] for _, row in available_rooms.iterrows()}
capacities[('dummy', '1')] = capacities[('HRZ','AMP')] + capacities[('KCK', '100')]
capacities[('dummy', '2')] = capacities[('DCH', '1055')] + capacities[('HRG', '100')]
capacities[('dummy', '3')] = capacities[('HRZ', '210')] + capacities[('HRZ', '212')]

# Student schedules (S): dict index -> set of CRNs
student_schedules = student_reg.groupby('PERSON_IDENTIFIER')['CRN'].apply(set).to_dict()

# Unique schedules and counts (S and N)
from collections import defaultdict
schedule_counts = defaultdict(int)
schedule_to_index = {}
index = 0
for sched in student_schedules.values():
    sched_tuple = frozenset(sched)
    if sched_tuple not in schedule_to_index:
        schedule_to_index[sched_tuple] = index
        index += 1
    schedule_counts[schedule_to_index[sched_tuple]] += 1

S = {idx: set(sched) for sched, idx in schedule_to_index.items()}
N = schedule_counts

print(f"Number of exams:             {len(E)}")
print(f"Number of timeslots:         {len(T)}")
print(f"Number of rooms:             {len(R)}")
print(f"Number of unique schedules:  {len(S)}")
print(f"Timeslots (chronological):   {T}")


Number of exams:             1389
Number of timeslots:         21
Number of rooms:             104
Number of unique schedules:  4708
Timeslots (chronological):   [('12/6/2023', 900), ('12/6/2023', 1400), ('12/6/2023', 1900), ('12/7/2023', 900), ('12/7/2023', 1400), ('12/7/2023', 1900), ('12/8/2023', 900), ('12/8/2023', 1400), ('12/8/2023', 1900), ('12/9/2023', 900), ('12/9/2023', 1400), ('12/9/2023', 1900), ('12/10/2023', 900), ('12/10/2023', 1400), ('12/10/2023', 1900), ('12/11/2023', 900), ('12/11/2023', 1400), ('12/11/2023', 1900), ('12/12/2023', 900), ('12/12/2023', 1400), ('12/12/2023', 1900)]


In [55]:
# ── TEST MODE ────────────────────────────────────────────────────────────────
# Run this cell instead of the normal data-loading cell (Cell 2) to use the
# synthetic test dataset (~100 exams, 500 students). Expected solve time: ~5 min.
# Includes one oversized exam (10001, 108 students) that requires a dummy room.
# ─────────────────────────────────────────────────────────────────────────────

import pandas as pd
from datetime import datetime
from collections import defaultdict

BASE = '/Users/benfreidinger/Desktop/decisionlab26/model/test_data'
class_info  = pd.read_csv(f'{BASE}/Class_Info_Test.csv')
final_exams = pd.read_csv(f'{BASE}/Final_Exams_Test.csv')
schedule    = pd.read_csv(f'{BASE}/Schedule_Test.csv')
student_reg = pd.read_csv(f'{BASE}/Student_Registration_Test.csv')

E = list(set(final_exams['CRN'].tolist()))
class_sizes = student_reg.groupby('CRN').size().to_dict()

timeslots = final_exams[['EXAM_DATE', 'EXAM_TIME']].drop_duplicates()
T = [(row['EXAM_DATE'], row['EXAM_TIME']) for _, row in timeslots.iterrows()]
T = sorted(T, key=lambda t: (datetime.strptime(t[0], '%m/%d/%Y'), t[1]))

available_rooms = class_info[class_info['AVAILABLE_TO_SCHEDULE'] == 'Y']
rooms = available_rooms[['BUILDING_CODE', 'ROOM_NUMBER']].drop_duplicates()
R = [(row['BUILDING_CODE'], row['ROOM_NUMBER']) for _, row in rooms.iterrows()]
R = R + [('dummy', '1'), ('dummy', '2'), ('dummy', '3')]

capacities = {(row['BUILDING_CODE'], row['ROOM_NUMBER']): row['ROOM_CAPACITY']
              for _, row in available_rooms.iterrows()}
capacities[('dummy', '1')] = capacities[('HRZ', 'AMP')]  + capacities[('KCK', '100')]
capacities[('dummy', '2')] = capacities[('DCH', '1055')] + capacities[('HRG', '100')]
capacities[('dummy', '3')] = capacities[('HRZ', '210')]  + capacities[('HRZ', '212')]

student_schedules = student_reg.groupby('PERSON_IDENTIFIER')['CRN'].apply(set).to_dict()
schedule_counts = defaultdict(int)
schedule_to_index = {}
index = 0
for sched in student_schedules.values():
    sched_tuple = frozenset(sched)
    if sched_tuple not in schedule_to_index:
        schedule_to_index[sched_tuple] = index
        index += 1
    schedule_counts[schedule_to_index[sched_tuple]] += 1

S = {idx: set(sched) for sched, idx in schedule_to_index.items()}
N = schedule_counts

print(f"Number of exams:             {len(E)}")
print(f"Number of timeslots:         {len(T)}")
print(f"Number of rooms:             {len(R)}")
print(f"Number of unique schedules:  {len(S)}")
print(f"Large exam (10001) size:     {class_sizes.get(10001, 0)} students")
print(f"Max single room capacity:    {max(capacities[r] for r in R if r[0] != 'dummy')}")


Number of exams:             100
Number of timeslots:         21
Number of rooms:             24
Number of unique schedules:  500
Large exam (10001) size:     108 students
Max single room capacity:    55


In [56]:
import gurobipy as gp
from gurobipy import GRB
from collections import defaultdict
import time

env = gp.Env(empty=True)
env.setParam("OutputFlag", 0)
env.start()

def schedule_model(E, T, R, S, N, capacities, class_sizes):
    """
    Inputs:
    E - list of CRNs (exams to schedule)
    T - ordered list of (EXAM_DATE, EXAM_TIME) timeslot tuples, sorted chronologically
    R - list of (BUILDING_CODE, ROOM_NUMBER) room tuples
    S - dict: schedule_idx -> set of CRNs
    N - dict: schedule_idx -> number of students with that schedule
    capacities - dict: (BUILDING_CODE, ROOM_NUMBER) -> capacity
    class_sizes - dict: CRN -> number of enrolled students

    Note: Gurobi flattens tuple indices, so all variable accesses unpack T and R
    elements explicitly: x[crn, date, time, bldg, room] not x[crn, t, r].
    """
    t_start_total = time.time()

    m = gp.Model(env=env)

    E_set = set(E)
    sched_keys = list(S.keys())

    # CRNs per schedule that have a final exam
    exam_crns_by_sched = {
        idx: [crn for crn in crns if crn in E_set]
        for idx, crns in S.items()
    }

    # Compute student overlap counts efficiently:
    # iterate over schedules rather than all O(|E|^2) pairs
    overlap = defaultdict(int)
    for idx, crns in S.items():
        exam_crns = [crn for crn in crns if crn in E_set]
        for i in range(len(exam_crns)):
            for j in range(i + 1, len(exam_crns)):
                pair = (min(exam_crns[i], exam_crns[j]), max(exam_crns[i], exam_crns[j]))
                overlap[pair] += N[idx]

    # Only build z variables for pairs that actually share students (3.7% of all pairs)
    E_pairs = list(overlap.keys())
    print(f"[{time.time()-t_start_total:.1f}s] Nonzero-overlap pairs: {len(E_pairs):,} (vs {len(E)*(len(E)-1)//2:,} total)")

    # DECISION VARIABLES
    # Gurobi flattens tuple indices, so keys are stored flat:
    #   x -> (crn, date, time, bldg, room)
    #   y -> (crn, date, time)
    #   z -> (crn1, crn2, date, time)
    #   v, u -> (date, time, sched_idx)
    #   d -> (date, time, sched_idx)
    x = m.addVars(E, T, R, vtype=GRB.BINARY, name="x")
    y = m.addVars(E, T, vtype=GRB.BINARY, name="y")
    z = m.addVars(E_pairs, T, vtype=GRB.CONTINUOUS, name="z")
    print(f"[{time.time()-t_start_total:.1f}s] Variables added: {len(E)*len(T)*len(R):,} x, {len(E)*len(T):,} y, {len(E_pairs)*len(T):,} z")

    # Blocking constraints for dummy room pairs:
    # when any exam uses a dummy room at time t, both constituent physical rooms
    # must be free (at most one "event" can claim each physical room per slot)
    m.addConstrs(
        (gp.quicksum(x[e, *t, 'HRZ', 'AMP']    for e in E) +
         gp.quicksum(x[e, *t, 'dummy', '1']     for e in E) <= 1 for t in T),
        name="block_dummy1_hrz")
    m.addConstrs(
        (gp.quicksum(x[e, *t, 'KCK', '100']    for e in E) +
         gp.quicksum(x[e, *t, 'dummy', '1']     for e in E) <= 1 for t in T),
        name="block_dummy1_kck")
    m.addConstrs(
        (gp.quicksum(x[e, *t, 'DCH', '1055']   for e in E) +
         gp.quicksum(x[e, *t, 'dummy', '2']     for e in E) <= 1 for t in T),
        name="block_dummy2_dch")
    m.addConstrs(
        (gp.quicksum(x[e, *t, 'HRG', '100']    for e in E) +
         gp.quicksum(x[e, *t, 'dummy', '2']     for e in E) <= 1 for t in T),
        name="block_dummy2_hrg")
    m.addConstrs(
        (gp.quicksum(x[e, *t, 'HRZ', '210']    for e in E) +
         gp.quicksum(x[e, *t, 'dummy', '3']     for e in E) <= 1 for t in T),
        name="block_dummy3_hrz210")
    m.addConstrs(
        (gp.quicksum(x[e, *t, 'HRZ', '212']    for e in E) +
         gp.quicksum(x[e, *t, 'dummy', '3']     for e in E) <= 1 for t in T),
        name="block_dummy3_hrz212")
    print(f"[{time.time()-t_start_total:.1f}s] Dummy room blocking constraints added ({6*len(T):,} constraints)")

    # Timeslot weights: index 1...|T|, so earlier slots cost less (objective minimizes)
    w = {t: i + 1 for i, t in enumerate(T)}

    # TRACKING 3 EXAMS IN 48 HOURS
    # T has 3 slots/day; a 6-slot window covers exactly 48 hours.
    # v_def uses y (already = sum of x over rooms) instead of summing x over rooms
    # directly — 24x fewer terms per constraint.
    window_starts = T[:-5]
    v = m.addVars(window_starts, sched_keys, vtype=GRB.INTEGER, name="v")
    print(f"[{time.time()-t_start_total:.1f}s] v variables added ({len(window_starts)*len(sched_keys):,})")

    for t_idx, t_start in enumerate(window_starts):
        window = T[t_idx:t_idx + 6]
        for sched_idx in sched_keys:
            m.addConstr(
                v[*t_start, sched_idx] == gp.quicksum(
                    y[crn, *t]
                    for crn in exam_crns_by_sched[sched_idx]
                    for t in window
                ),
                name=f"v_def[{t_start},{sched_idx}]"
            )
    print(f"[{time.time()-t_start_total:.1f}s] 48h window (v_def) constraints added ({len(window_starts)*len(sched_keys):,} constraints)")

    # Big-M formulation replaces addGenConstrIndicator (faster to build).
    # Minimizing u means the solver sets u=1 only when v>=3.
    u = m.addVars(window_starts, sched_keys, vtype=GRB.BINARY, name="u")
    for t_start in window_starts:
        for sched_idx in sched_keys:
            n_exams = len(exam_crns_by_sched[sched_idx])
            m.addConstr(v[*t_start, sched_idx] <= 2 + n_exams * u[*t_start, sched_idx])
    print(f"[{time.time()-t_start_total:.1f}s] Big-M (u) constraints added ({len(window_starts)*len(sched_keys):,} constraints)")

    # TRACKING BACK-TO-BACK EXAMS
    # Use y directly instead of an intermediate p variable — avoids ~535K extra constraints.
    consec_starts = T[:-1]
    d = m.addVars(consec_starts, sched_keys, vtype=GRB.BINARY, name="d")
    for t_idx, t in enumerate(consec_starts):
        t_next = T[t_idx + 1]
        for sched_idx in sched_keys:
            crns = exam_crns_by_sched[sched_idx]
            if crns:
                at_t      = gp.quicksum(y[crn, *t]      for crn in crns)
                at_t_next = gp.quicksum(y[crn, *t_next] for crn in crns)
                m.addConstr(d[*t, sched_idx] <= at_t)
                m.addConstr(d[*t, sched_idx] <= at_t_next)
                m.addConstr(d[*t, sched_idx] >= at_t + at_t_next - 1)
            else:
                m.addConstr(d[*t, sched_idx] == 0)
    print(f"[{time.time()-t_start_total:.1f}s] Back-to-back (d) constraints added ({3*len(consec_starts)*len(sched_keys):,} constraints)")

    # OBJECTIVE
    lamb = 1
    mu   = 10  # penalty for 3 exams in 48 hours
    nu   = 5   # penalty for back-to-back

    m.setObjective(
        lamb * gp.quicksum(overlap[(c1, c2)] * z[c1, c2, *t] for c1, c2 in E_pairs for t in T) +
        mu   * gp.quicksum(u[*t, idx]  for t in window_starts for idx in sched_keys) +
        nu   * gp.quicksum(d[*t, idx]  for t in consec_starts  for idx in sched_keys),
        GRB.MINIMIZE
    )
    print(f"[{time.time()-t_start_total:.1f}s] Objective set")

    # HARD CONSTRAINTS

    # Each exam scheduled exactly once
    m.addConstrs(
        (gp.quicksum(x[crn, *t, *r] for t in T for r in R) == 1 for crn in E),
        name="assign_once"
    )
    print(f"[{time.time()-t_start_total:.1f}s] assign_once constraints added ({len(E):,} constraints)")

    # Room capacities
    m.addConstrs(
        (gp.quicksum(class_sizes[crn] * x[crn, *t, *r] for crn in E) <= capacities[r]
         for t in T for r in R),
        name="capacity"
    )
    print(f"[{time.time()-t_start_total:.1f}s] Capacity constraints added ({len(T)*len(R):,} constraints)")

    # y[crn, t] = 1 iff crn is scheduled at t
    m.addConstrs(
        (y[crn, *t] == gp.quicksum(x[crn, *t, *r] for r in R) for crn in E for t in T),
        name="y_def"
    )
    print(f"[{time.time()-t_start_total:.1f}s] y_def constraints added ({len(E)*len(T):,} constraints)")

    # Linearize z[c1, c2, t] = y[c1, t] * y[c2, t]
    m.addConstrs((z[c1, c2, *t] <= y[c1, *t]                    for c1, c2 in E_pairs for t in T))
    m.addConstrs((z[c1, c2, *t] <= y[c2, *t]                    for c1, c2 in E_pairs for t in T))
    m.addConstrs((z[c1, c2, *t] >= y[c1, *t] + y[c2, *t] - 1   for c1, c2 in E_pairs for t in T))
    print(f"[{time.time()-t_start_total:.1f}s] z linearization constraints added ({3*len(E_pairs)*len(T):,} constraints)")

    print(f"\n[{time.time()-t_start_total:.1f}s] Model built. Starting optimization...")
    print(f"    Variables: {m.NumVars:,}  |  Constraints: {m.NumConstrs:,}  |  Binary vars: {m.NumBinVars:,}")

    t_solve = time.time()
    m.optimize()
    print(f"[{time.time()-t_start_total:.1f}s] Optimization complete (solve time: {time.time()-t_solve:.1f}s)")

    if m.Status in (GRB.OPTIMAL, GRB.SUBOPTIMAL):
        print(f"Objective value: {m.ObjVal:.2f}")
        schedule_result = {}
        for crn in E:
            for t in T:
                for r in R:
                    if x[crn, *t, *r].X > 0.5:
                        schedule_result[crn] = {"timeslot": t, "room": r}
        return schedule_result
    else:
        print(f"No optimal solution found. Status: {m.Status}")
        return None


In [57]:
import json as _json

result = schedule_model(E, T, R, S, N, capacities, class_sizes)

if result is not None:
    # Convert tuple keys to serializable strings
    output = {
        str(crn): {
            'date': v['timeslot'][0],
            'time': v['timeslot'][1],
            'building': v['room'][0],
            'room': v['room'][1]
        }
        for crn, v in result.items()
    }
    out_path = '/Users/benfreidinger/Desktop/decisionlab26/exam_schedule_optimized.json'
    with open(out_path, 'w') as f:
        _json.dump(output, f, indent=2)
    print(f'Schedule written to {out_path}')


[0.0s] Nonzero-overlap pairs: 2,311 (vs 4,950 total)
[0.4s] Variables added: 50,400 x, 2,100 y, 48,531 z
[0.4s] Dummy room blocking constraints added (126 constraints)
[0.4s] v variables added (8,000)
[0.6s] 48h window (v_def) constraints added (8,000 constraints)
[0.6s] Big-M (u) constraints added (8,000 constraints)
[1.0s] Back-to-back (d) constraints added (30,000 constraints)
[1.0s] Objective set
[1.0s] assign_once constraints added (100 constraints)
[1.1s] Capacity constraints added (504 constraints)
[1.2s] y_def constraints added (2,100 constraints)
[2.5s] z linearization constraints added (145,593 constraints)

[2.5s] Model built. Starting optimization...
    Variables: 0  |  Constraints: 0  |  Binary vars: 0

Interrupt request received
[745.6s] Optimization complete (solve time: 743.1s)
No optimal solution found. Status: 11
